# A/B Test Simulation

The dataset has no native experiment flag, so we construct an **as-if** experiment (spec §7.3): pick two comparable segments and treat one as control, one as treatment.

Approach used here: compare two customer **states with similar order volume and product-category mix**, testing whether their average delivery time differs. (An alternative — comparing two carriers — isn't available: this dataset records *when* a package was handed to the carrier, not *which* carrier, so shipping-company comparisons can't be built from it. Logged in PROBLEMS.md.)

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = PROJECT_ROOT / "data" / "processed" / "olist.db"
conn = sqlite3.connect(DB_PATH)

def q(sql):
    return pd.read_sql(sql, conn)
from scipy import stats


## 1. Pick two comparable states

In [2]:
regional = q("SELECT * FROM v_regional_analysis WHERE orders >= 200 ORDER BY orders DESC")
regional

,customer_state,orders,aov,avg_freight,freight_ratio,avg_delivery_days
0,sp,40501,125.12,15.12,0.139,8.7
1,rj,12350,142.48,20.91,0.168,15.1
2,mg,11354,136.73,20.63,0.172,12.0
3,rs,5345,136.37,21.61,0.182,15.2
4,pr,4923,135.30,20.47,0.174,11.9
5,sc,3546,142.98,21.51,0.174,15.0
6,ba,3256,151.59,26.49,0.198,19.2
7,df,2080,142.55,21.07,0.167,13.0
8,es,1995,134.66,22.03,0.182,15.6
9,go,1957,144.53,22.56,0.182,15.4


In [3]:
# Find the pair of states with the closest order volume AND closest freight_ratio
# (as a rough proxy for "similar product mix"), among states with enough volume
# to give the t-test reasonable power.
from itertools import combinations

candidates = regional.set_index("customer_state")
pairs = []
for a, b in combinations(candidates.index, 2):
    vol_diff = abs(candidates.loc[a, "orders"] - candidates.loc[b, "orders"]) / max(candidates.loc[a, "orders"], candidates.loc[b, "orders"])
    mix_diff = abs(candidates.loc[a, "freight_ratio"] - candidates.loc[b, "freight_ratio"])
    pairs.append((a, b, vol_diff, mix_diff))

pairs_df = pd.DataFrame(pairs, columns=["state_a", "state_b", "vol_diff_pct", "mix_diff"])
pairs_df.sort_values(["vol_diff_pct", "mix_diff"]).head(10)

,state_a,state_b,vol_diff_pct,mix_diff
238,pi,rn,0.004202,0.014
148,es,go,0.019048,0.000
217,ma,ms,0.022315,0.099
133,df,es,0.040865,0.015
134,df,go,0.059135,0.015
198,pa,mt,0.063425,0.024
63,rs,pr,0.078952,0.008
232,pb,pi,0.079304,0.017
22,rj,mg,0.080648,0.004
100,sc,ba,0.081782,0.024


**Pick the top row above** (closest volume + closest freight ratio) as control/treatment. Set `STATE_CONTROL` / `STATE_TREATMENT` below once you've run this on real data.

In [4]:
STATE_CONTROL = "pi"   # chosen: closest order-volume + freight-mix match to rn (see pairs_df above)
STATE_TREATMENT = "rn"

orders = q(f"""
    SELECT o.order_id, c.customer_state,
           julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp) AS delivery_days,
           r.review_score
    FROM stg_orders o
    JOIN stg_customers c ON c.customer_id = o.customer_id
    LEFT JOIN stg_order_reviews r ON r.order_id = o.order_id
    WHERE o.order_status = 'delivered'
      AND c.customer_state IN ('{STATE_CONTROL}', '{STATE_TREATMENT}')
""")
control = orders.loc[orders.customer_state == STATE_CONTROL, "delivery_days"].dropna()
treatment = orders.loc[orders.customer_state == STATE_TREATMENT, "delivery_days"].dropna()
len(control), len(treatment)

(476, 474)

## 2. Two-sample hypothesis test

In [5]:
t_stat, p_value = stats.ttest_ind(treatment, control, equal_var=False)
effect_size = treatment.mean() - control.mean()

# 95% CI for the difference in means
se = np.sqrt(treatment.var(ddof=1) / len(treatment) + control.var(ddof=1) / len(control))
ci_low, ci_high = effect_size - 1.96 * se, effect_size + 1.96 * se

print(f"Control ({STATE_CONTROL}) mean delivery days: {control.mean():.2f}")
print(f"Treatment ({STATE_TREATMENT}) mean delivery days: {treatment.mean():.2f}")
print(f"Effect size (treatment - control): {effect_size:.2f} days")
print(f"95% CI: [{ci_low:.2f}, {ci_high:.2f}]")
print(f"t = {t_stat:.3f}, p = {p_value:.4f}")

Control (pi) mean delivery days: 19.46
Treatment (rn) mean delivery days: 19.28
Effect size (treatment - control): -0.18 days
95% CI: [-1.94, 1.59]
t = -0.199, p = 0.8426


## 3. Ship / no-ship recommendation

_Fill in after running on real data — explicitly separate the two questions:_

- **Statistical significance:** is p < 0.05? (i.e., is this effect likely real, not noise?)
- **Practical significance:** is the effect size (in days) big enough to matter to a customer or the business, regardless of significance? A 0.3-day difference can be "statistically significant" with enough orders and still not be worth acting on.

**Recommendation:** [ship / no-ship / needs more data], because ___.